# Temporal risk ranking for hypotension (public data)

Compare gradient boosting on a snapshot of six latest vitals with a model that adds each vital's minimum, mean, variation, slope and observation count. Hold out 25% of stays per seed and evaluate AUROC, average precision and recall in the top 10% of risk scores. This transfers the time-bounded EHR risk-ranking idea from Wiens's CDI study to a different outcome. Use public PhysioNet Challenge 2012 Set A. Exclude stays with observed systolic pressure below 90 mmHg before the six-hour landmark. Require at least one systolic measurement on each side of the landmark. Use only measurements through hour six as predictors, and define the target as any observed systolic reading below 90 mmHg during hours 6–12. The outcome depends on recorded measurements and the splits are retrospective.

In [ ]:
from pathlib import Path
import subprocess, sys
root = Path.cwd() / 'temporal-hypotension-ranking-real-data-run'
root.mkdir(exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install',*['numpy==2.2.6', 'scipy==1.15.3', 'scikit-learn==1.7.2', 'joblib==1.6.0']],check=True)

In [ ]:
(root / 'prepare.py').write_text('import os\nfrom pathlib import Path\nfrom urllib.request import urlretrieve\n\nroot = (\n    Path(os.environ.get("PORTFOLIO_DATA", Path(__file__).resolve().parent / "data"))\n    / "icu2012"\n)\nroot.mkdir(parents=True, exist_ok=True)\nfor name in ["set-a.tar.gz", "Outcomes-a.txt"]:\n    target = root / name\n    if not target.exists():\n        temp = target.with_suffix(".part")\n        urlretrieve("https://physionet.org/files/challenge-2012/1.0.0/" + name, temp)\n        temp.replace(target)\n',encoding='utf8')
subprocess.run([sys.executable,'prepare.py'],cwd=root,check=True)

In [ ]:
(root / 'train.py').write_text('import json\nimport os\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nfrom sklearn.metrics import (\n    average_precision_score,\n    roc_auc_score,\n)\n\nROOT = Path(__file__).resolve().parent\nDATA = Path(os.environ.get("PORTFOLIO_DATA", ROOT / "data"))\nSEEDS = [13, 37, 61]\n\n\ndef save(obj, name):\n    folder = ROOT / "checkpoints"\n    folder.mkdir(exist_ok=True)\n    joblib.dump(obj, folder / (name + ".joblib"), compress=3)\n\n\ndef scores(y, p):\n    return {\n        "auroc": float(roc_auc_score(y, p)),\n        "average_precision": float(average_precision_score(y, p)),\n    }\n\n\nfrom sklearn.ensemble import HistGradientBoostingClassifier\nfrom sklearn.model_selection import train_test_split\n\nVITALS = ("SysABP", "NISysABP", "MAP", "HR", "RespRate", "SpO2")\n\n\ndef summarize(early):\n    latest = []\n    summary = []\n    for name in VITALS:\n        values = sorted(early.get(name, []))\n        if values:\n            t, z = np.array(values).T\n            latest.append(z[-1])\n            summary.extend(\n                [\n                    z.min(),\n                    z.mean(),\n                    z.std(),\n                    (z[-1] - z[0]) / max(t[-1] - t[0], 0.25),\n                    len(z),\n                ]\n            )\n        else:\n            latest.append(np.nan)\n            summary.extend([np.nan] * 4 + [0])\n    return latest + summary\n\n\ndef run(seed):\n    x, y = dataset(seed)\n    train, test = train_test_split(\n        np.arange(len(y)), test_size=0.25, stratify=y, random_state=seed\n    )\n    results = {}\n    for method, cols in [("snapshot", len(VITALS)), ("temporal", x.shape[1])]:\n        fit = HistGradientBoostingClassifier(\n            max_iter=100,\n            max_leaf_nodes=12,\n            min_samples_leaf=24,\n            l2_regularization=3,\n            random_state=seed,\n        )\n        fit.fit(x[train, :cols], y[train])\n        p = fit.predict_proba(x[test, :cols])[:, 1]\n        order = np.argsort(-p)[: len(test) // 10]\n        results[method] = {\n            **scores(y[test], p),\n            "recall_top_10pct": float(y[test][order].sum() / y[test].sum()),\n            "eligible_stays": len(y),\n            "events": int(y.sum()),\n        }\n        save(fit, f"{method}-{seed}")\n    return results\n\n\nimport csv\nimport io\nimport tarfile\n\n\ndef dataset(seed):\n    x = []\n    labels = []\n    with tarfile.open(DATA / "icu2012/set-a.tar.gz") as z:\n        for member in z:\n            if not member.isfile() or not Path(member.name).stem.isdigit():\n                continue\n            early = {}\n            future = []\n            with io.TextIOWrapper(z.extractfile(member)) as stream:\n                for row in csv.DictReader(stream):\n                    name = row["Parameter"]\n                    if name not in VITALS:\n                        continue\n                    h, m = map(int, row["Time"].split(":"))\n                    stamp = h + m / 60\n                    value = float(row["Value"])\n                    if value <= 0:\n                        continue\n                    if name in VITALS[:2] and not 40 <= value <= 280:\n                        continue\n                    if stamp <= 6:\n                        early.setdefault(name, []).append((stamp, value))\n                    elif stamp <= 12 and name in VITALS[:2]:\n                        future.append(value)\n            previous = [v for k in VITALS[:2] for _, v in early.get(k, [])]\n            if not previous or not future or min(previous) < 90:\n                continue\n            x.append(summarize(early))\n            labels.append(int(min(future) < 90))\n    return np.asarray(x), np.asarray(labels)\n\n\nif __name__ == "__main__":\n    runs = {s: run(s) for s in SEEDS}\n    first = next(iter(runs.values()))\n    means = {\n        k: {m: float(np.mean([r[k][m] for r in runs.values()])) for m in v}\n        for k, v in first.items()\n    }\n    (ROOT / "results.json").write_text(\n        json.dumps({"seeds": SEEDS, "runs": runs, "means": means}, indent=2)\n    )\n    print(means)\n',encoding='utf8')
subprocess.run([sys.executable,'train.py'],cwd=root,check=True)

In [ ]:
print((root/'results.json').read_text())